In [1]:
# load dungeon data

import json
import random
from pathlib import Path

data_path = Path("../datasets/dungeon_10k_4_8_3_5_mkr.jsonl")
with open(data_path) as f:
    data = [json.loads(line) for line in f]

# strip _id fields
for entry in data:
    if "_id" in entry:
        del entry["_id"]

# Split into train/eval sets
random.shuffle(data)
split_idx = int(0.8 * len(data))
train_data = data[:split_idx]
eval_data = data[split_idx:]

print(f"Train set: {len(train_data)} records")
print(f"Eval set: {len(eval_data)} records")

# print first data point
print(json.dumps(data[0], indent=2))

Train set: 8000 records
Eval set: 2000 records
{
  "door": 1,
  "key_color": "green",
  "corridor": [
    {
      "door_no": 3,
      "red_key": "diamonds",
      "blue_key": "diamonds",
      "green_key": "spellbooks"
    },
    {
      "monsters": [
        "orc"
      ],
      "door_no": 0,
      "blue_key": "gemstones",
      "green_key": "artifacts",
      "red_key": "diamonds"
    },
    {
      "monsters": [
        "dragon"
      ],
      "door_no": 2,
      "blue_key": "gemstones",
      "green_key": "spellbooks",
      "red_key": "gold"
    },
    {
      "monsters": [
        "orc",
        "wolf"
      ],
      "door_no": 1,
      "green_key": "gold",
      "blue_key": "gold",
      "red_key": "diamonds"
    },
    {
      "monsters": [
        "goblin"
      ],
      "door_no": 4,
      "red_key": "gemstones",
      "green_key": "artifacts",
      "blue_key": "diamonds"
    }
  ],
  "treasure": "gold"
}


In [ ]:
from origami import ModelConfig, OrigamiConfig, OrigamiPipeline, TrainingConfig
from origami.training import TableLogCallback, accuracy

config = OrigamiConfig(
    model=ModelConfig(
        d_model=192,
        n_heads=8,
        n_layers=6,
        d_ff=784,
        dropout=0.0,
        use_grammar_constraints=True,
    ),
    training=TrainingConfig(
        shuffle_keys=False,
        batch_size=100,
        warmup_steps=1000,
        learning_rate=5e-4,
        eval_strategy="epoch",
        eval_epochs=5,
        eval_metrics={"acc": accuracy},
        eval_sample_size=100,
        target_key="treasure",
    ),
)

pipeline = OrigamiPipeline(config)
pipeline.fit(
    train_data,
    eval_data=eval_data,
    callbacks=[TableLogCallback(print_every=50)],
    epochs=50,
    verbose=True,
)

Vocabulary size: 40
Model parameters: 2,771,080
Training device: mps
| step: 50 | epoch: 0 | lr: 2.50e-05 | batch_dt:  140ms | loss: 2.3798 |
| step: 100 | epoch: 1 | lr: 5.00e-05 | batch_dt:  151ms | loss: 1.3637 |
| step: 150 | epoch: 1 | lr: 7.50e-05 | batch_dt:  151ms | loss: 1.0836 |
| step: 200 | epoch: 2 | lr: 1.00e-04 | batch_dt:  167ms | loss: 0.9698 |
| step: 250 | epoch: 3 | lr: 1.25e-04 | batch_dt:  157ms | loss: 0.8629 |
| step: 300 | epoch: 3 | lr: 1.50e-04 | batch_dt:  147ms | loss: 0.8384 |
| step: 350 | epoch: 4 | lr: 1.75e-04 | batch_dt:  147ms | loss: 0.8274 |
| step: 400 | epoch: 4 | lr: 2.00e-04 | batch_dt:  146ms | loss: 0.8230 |
| step: 450 | epoch: 5 | lr: 2.25e-04 | batch_dt:  154ms | loss: 0.8073 | val_acc: 0.1800 | val_loss: 0.8190 |
| step: 500 | epoch: 6 | lr: 2.50e-04 | batch_dt:  150ms | loss: 0.8142 |
| step: 550 | epoch: 6 | lr: 2.75e-04 | batch_dt:  149ms | loss: 0.8062 |
| step: 600 | epoch: 7 | lr: 3.00e-04 | batch_dt:  146ms | loss: 0.7874 |
| step:

In [ ]:
# pipeline.save("dungeon_pipeline.pt")

In [ ]:
from origami import OrigamiPipeline

# pipeline = OrigamiPipeline.load("dungeon_pipeline.pt")

In [ ]:
from origami.training import accuracy

pipeline.evaluate(eval_data, metrics={"acc": accuracy})

In [ ]:
doc = pipeline.generate(1)[0]

print(json.dumps(doc, indent=2))